In [12]:
import pandas as pd
import os
import csv
import json

In [13]:
file_path = os.path.join(os.getcwd(), "data/new_data_final.csv")
df = pd.read_csv(file_path, encoding="utf-8")
df.head(1)

,id,name,address,lat,lon,poi_type,avg_stars,total_reviews,stay_time,normalize_stars_reviews,crowd,offerings,atmosphere,highlights,dining_options,children,accessibility,popular_for,opening_hours
0,0f9d2009-9436-46a4-b354-b0261898a39e,The Pub Coffee - Beer & Cocktail,"18A17 Tăng Nhơn Phú, Phước Long B, Quận 9, Thà...",10.829481,106.773785,"Cafe,Bar",4.9,181,30,0.755,Groups,"Alcohol, Beer, Cocktails, Coffee, Hard liquor","Casual, Cozy","Great beer selection, Great coffee, Live music...",Table service,NaN,NaN,NaN,"[{'day': 'Monday', 'hours': [{'start': '00:00'..."


## Extract Features (Columns) from DataFrame

In [14]:
# Get all column names (features)
features = df.columns.tolist()
print("Column Features:")
print(features)
print(f"\nTotal number of features: {len(features)}")

Column Features:
['id', 'name', 'address', 'lat', 'lon', 'poi_type', 'avg_stars', 'total_reviews', 'stay_time', 'normalize_stars_reviews', 'crowd', 'offerings', 'atmosphere', 'highlights', 'dining_options', 'children', 'accessibility', 'popular_for', 'opening_hours']

Total number of features: 19


In [15]:
# Display features with their data types
print("Features with Data Types:")
print(df.dtypes)

Features with Data Types:
id                          object
name                        object
address                     object
lat                        float64
lon                        float64
poi_type                    object
avg_stars                  float64
total_reviews                int64
stay_time                    int64
normalize_stars_reviews    float64
crowd                       object
offerings                   object
atmosphere                  object
highlights                  object
dining_options              object
children                    object
accessibility               object
popular_for                 object
opening_hours               object
dtype: object


In [16]:
# Display features with missing values count
print("Features with Missing Values Count:")
print(df.isnull().sum())

Features with Missing Values Count:
id                            0
name                          0
address                       0
lat                           0
lon                           0
poi_type                      0
avg_stars                     0
total_reviews                 0
stay_time                     0
normalize_stars_reviews       0
crowd                       805
offerings                   773
atmosphere                  893
highlights                  850
dining_options              893
children                    968
accessibility               801
popular_for                1157
opening_hours                 0
dtype: int64


## POI Types with Empty Opening Hours

In [17]:
# Check data type of opening_hours column
print("Data type of opening_hours:", df['opening_hours'].dtype)
print("\nSample values:")
print(df['opening_hours'].head(10))

Data type of opening_hours: object

Sample values:
0    [{'day': 'Monday', 'hours': [{'start': '00:00'...
1    [{'day': 'Monday', 'hours': [{'start': '08:00'...
2    [{'day': 'Tuesday', 'hours': [{'start': '11:00...
3    [{'day': 'Tuesday', 'hours': [{'start': '17:30...
4    [{'day': 'Monday', 'hours': [{'start': '10:00'...
5    [{'day': 'Monday', 'hours': [{'start': '16:00'...
6    [{'day': 'Monday', 'hours': [{'start': '07:00'...
7    [{'day': 'Monday', 'hours': [{'start': '07:00'...
8    [{'day': 'Monday', 'hours': [{'start': '13:30'...
9    [{'day': 'Tuesday', 'hours': [{'start': '22:00...
Name: opening_hours, dtype: object


In [18]:
# Filter POI types with empty opening_hours
# Check for empty lists/strings or null values
empty_hours_mask = (df['opening_hours'] == '[]') | (df['opening_hours'] == '') | (df['opening_hours'].isna())
poi_types_with_empty_hours = df[empty_hours_mask]['poi_type'].unique()

print(f"POI Types with Empty Opening Hours ({len(poi_types_with_empty_hours)} types):")
print("="*50)
for poi_type in sorted(poi_types_with_empty_hours):
    count = len(df[(empty_hours_mask) & (df['poi_type'] == poi_type)])
    print(f"{poi_type}: {count} entries")

print("="*50)
print(f"\nTotal entries with empty opening_hours: {empty_hours_mask.sum()}")

POI Types with Empty Opening Hours (118 types):
Airport: 2 entries
Airport shuttle service,Minibus taxi service,Taxi service: 1 entries
Airport,Travel lounge: 1 entries
Amusement park: 1 entries
Andalusian restaurant,Spanish restaurant: 1 entries
Art center: 4 entries
Art gallery,Cultural center,Photography studio,Tourist attraction: 1 entries
Art museum: 1 entries
Art school,Art gallery,Concert hall,Dance hall,Rehearsal studio,Showroom,Sports club: 1 entries
Art studio,Artistic handicrafts,Sculpture,Sculpture museum,Statuary: 1 entries
Auto repair shop,Auto parts market,Car wash: 1 entries
Bakery: 5 entries
Bakery,Hotel: 1 entries
Baptist church,Church,Evangelical church: 1 entries
Bar: 3 entries
Bar & grill: 1 entries
Bar,Lounge: 1 entries
Beach: 13 entries
Beach clothing store: 1 entries
Beach pavillion: 1 entries
Beach volleyball court: 1 entries
Boarding house,Cafe: 1 entries
Brewpub,Bar,Bar & grill,Restaurant,Sandwich shop,Snack bar: 1 entries
Brewpub,Bar,Brewery,Restaurant: 1 en

## Fill Empty Opening Hours with Default Values

In [19]:
import json
# Define default opening hours (24/7)
default_opening_hours = [
    {'day': 'Monday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
    {'day': 'Tuesday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
    {'day': 'Wednesday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
    {'day': 'Thursday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
    {'day': 'Friday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
    {'day': 'Saturday', 'hours': [{'start': '00:00', 'end': '23:59'}]},
    {'day': 'Sunday', 'hours': [{'start': '00:00', 'end': '23:59'}]}
]

default_opening_hours_str = json.dumps(default_opening_hours)
print("Default opening hours:")
print(default_opening_hours_str)

Default opening hours:
[{"day": "Monday", "hours": [{"start": "00:00", "end": "23:59"}]}, {"day": "Tuesday", "hours": [{"start": "00:00", "end": "23:59"}]}, {"day": "Wednesday", "hours": [{"start": "00:00", "end": "23:59"}]}, {"day": "Thursday", "hours": [{"start": "00:00", "end": "23:59"}]}, {"day": "Friday", "hours": [{"start": "00:00", "end": "23:59"}]}, {"day": "Saturday", "hours": [{"start": "00:00", "end": "23:59"}]}, {"day": "Sunday", "hours": [{"start": "00:00", "end": "23:59"}]}]


In [22]:
# Fill empty opening_hours with default values
df.loc[empty_hours_mask, 'opening_hours'] = default_opening_hours_str

print(f"Updated {empty_hours_mask.sum()} entries with default opening hours")
print("\nSample of updated entries:")
print(df[df['poi_type'] == 'airport'][['poi_type', 'opening_hours']].head())

Updated 317 entries with default opening hours

Sample of updated entries:
Empty DataFrame
Columns: [poi_type, opening_hours]
Index: []


In [23]:
# Save the updated dataframe to a new CSV file
output_path = os.path.join(os.getcwd(), "data/new_data_final_filled.csv")
df.to_csv(output_path, index=False, encoding='utf-8')
print(f"\nDataFrame saved to: {output_path}")
print(f"Total rows: {len(df)}")


DataFrame saved to: c:\Users\nguye\Desktop\vinamo\Kyanon-support-localtion\scripts\clean_data\data/new_data_final_filled.csv
Total rows: 1454
